# RSX-V2-13 · Scoring, leakage and protocol audit (CPU)

**Runs without GPU, Drive or paid APIs.** Audits historical E10/E11 and the four disputed cases, and checks that 48 new calibration and 60 new held-out problems are distinct. Recreates the exact frozen protocol files. This does **not** constitute human scoring adjudication; the human-review CSV intentionally remains unreviewed.

In [ ]:

import base64,zlib,json,hashlib,csv,io,zipfile,os,statistics
from pathlib import Path
import pandas as pd
from IPython.display import display
DATA=json.loads(zlib.decompress(base64.b64decode('eNrVfVtvJMtx5l9pzJNtcaiMiIy8HGNhaG1514DsB8tYL+BjDPI6Qx8Om2KT52gk+H2f9yfuL9mIKl66m82uqpkeghYkcborOysrLl98kZmV8ed3JV1e5Jt0e7G+evfD6t/+/O6ni6sq/3qXbi5uP31utxfl3dnq3dXd59xuNtrk3+Xj9c06X7bPH/rF1cd2c31zcXWrvwmldUZvAHu3GVsk9iEWglSCzYEbe2vAZE/GQLGdWuyOe7DJ+9Syf7fV9cUwipvNH9//jD9sjfKHp4H9YIy5/8nn62EAv19f/txWt5/aqq8vL9e/yPBW49BXt2nz0/nqb9NlubtMt2Ojzd3n1bqvgl+lq7pCOL/vbd21MzBBP2+uLy+GzrdFJd/fppuPTS9IO/3YPl9rx3st3z8N9715959nq2+TcCeLtbPNrfXke80peaweCJ0hn4NcrYFtB84GPDtXRdIu1ILcK4e8XMLwTRKWH9a7cqtSdjBIGcK+lONcMceZcoZvlzM733xMrubsgQk4Rk4mVU4Wavc9NYyeEsZge/UBW7di510UUTIzwnI540I5/7083XC9XvTebtpVaatfPrWrFdHqYiOmnW9vUrltddWl0xW5XbHTLJnTPIHjtws8J1NiUtCAUNFV6gwiWHat2hipRmgAthUA1yNzRFegB2hi5ZRS5OUCp4UC/7uLny9qW3nGVf6yAjdY88eL+x/94W59e9GubnelbP0sMVs/T8707XJOptcmgGvJUKfQTffGe+8wOouBTYzIrmPClIN3ppnk2RpvUy4UoNvlcrangGiOg7jjrnjdPORwr4fPUFpC57nnEEysZLoprVQvxsxZABvYNIgi7lwq5VxzjL4aWzwVE2o2y8XLp8Jni2MUtPv4PDsMmtfD52yYU60xumwltkXyYsalYbLV15Cq4eZ9LgIkLsRuSDAlUsjJFVugpLhczu5U+AwH8dntmTa7WUJn92oIXZpIMoZmbQTqgh3WhpQzm9qbs6bWjFy9tLCAUE3j2sQDMDFYsg5wucj91yG0kiGFaJwF0QzzBA2vBtGmBjQ+5ooVWYRKKl8BkGaFAlUgXzMFkx1TxNQFoiVKJq6RCymsfAVEh1NAtDMjvzO78vVmlny9eTWMJoem+pYc1CrMITVPUFMQpCBJTEr2pTc0uQhuROzWOuuKSLwHU5tQu6/A6HgqjA40yngfo8HhPIyWhq+G0cW3IHlfMz04I9yBHRSooQkYuxxbZkkTQwJ2ZNmLMjJ2oljE/H2XmFkXyxnMyTA6HMRo2mN3PI/d8athdIyxesHlXIqPvWVqqVqHJXHnJvwOwDgWQp2CkOqQbQgC2YHkT5SL8StEDl/JouPAohHnQDTNi4XkXg2iuaaUqouoeTV7yRQzGTaQRcrekdCRLMkJcEEBksSFrWdjiqUshM12Wi5nPAVE0z184He14xNAtLPFhUqCBCV0G4Ah2mhQsj2BYaF1Egqj8I5ue7Ot+0JOeCAr6SNrLC2fSAI6FUTHkXUg78oYrZkXCLXhq0E0QpD0Ayh5YRg1N2EXvUVXonUcLBeRv9AwxeRGnp2vgh6+OUnqxJpLD8vlbE82zYGHIDruTXM4Oy9DtK830SGpNNqIkErNnCgjVp84+57YpxiNc73kVkKJYsyl2BgkgKYeWysmpuXsA/grabQhOx+jv+9MR1nfXd3W9S9Xe3IGyThBmDmGIxKvgawkgyZgLNhbsySYza2FEC27gCGlQKahfGeL8L/mk/cFi/AVEIB3syT+OMLnk9J/+3BpdX33pz9dth9WdxsV+tPgV+2PYsOXX1ZrMe+zVb67uByNfpTbKgCufhHZrH51tnp/tvqrs9WvB51cJ3EIabdpm/PV7z+tf1ml1c8yqCodXt+0zUZGdn+zlsqnQYdXD8revuf56n88qPf+lmmz+vt/+Kff/O7Db/7p9//623/eVfZfCD/6lYz9L1d/JaOfpXh5hCOaf5TefvB4SfHxbCXRCN2xNEpCSHaErYsvGaGgJpqQowRiV5Mo2fUozCh2YZtoRfUuQzZN2nXju2VarHaYp/aHoU8p3cOb0nkUXbNoHeeRMQ+z1A3z1I0kEhMDQj4298ZEgUuJEXwRThATWKq1ZeFmJQldEHfuziYEyTa8aeSiFV5WsUfM4M1iheM8hT8NfkrlwnTelM6R1MFx9V7GPg/ewzy142wvl3gB5hgTD53AN8IUITQvzKWVnFuLtdrencngUvXojK0sAFB9bAgCDAacSYbLYqXTbC8fhj6lcsGBN4XsUVzc/6Wi+0y66mYpnGYrXDs8qnHwQUgSWSpgJQFgloQXXOdeW6fqXWi9Q+3UoyQJNrFkatn57kLnkgQgFmvcztb4OPYplRObt6dzdBrMZypdnuCUwXwASKMgeYzFYRafNpIHUgsQhBlX9MELl6OCgUTdyaPO6ZnWUDh08JF6NViTo8R1OYvjBeg+DH4a3fENortR5eNMdMdTBnWQpAum8N2nXnRKxYaSuPXcs0uth1pc985RbsZkjJJJCXfzzQsaCAr0CsLvU3MlLVa7m0neHwc/ifDC79+S2sGqo3sJ6nMhHsMpg/rgLDyhdvFa4e+uGSw2A0dCCwkk2LcaU6yRM3PMEKohjwF6ddQQmyEj9N34tljtfqa3Pw5+Su3MbwvkR0fnBaGd+ZShXfM/jY/HGHxloCSa7uK4SXga2RhLtsLZAXIWZPfcMYiTG0bnm26FkkBfapdPJVRcrPUw09kfxj6ldIhvLLQDPsb2eQxenuCkod1Oax0z22iraDjaQuLu3tbQvaTsxXuoxcSCDiSFQ6KUe8zkAzXheKkBBl7u63Gmr9uZWncY31ZgV4SXxOJXM3Uu4z9psi5hUcQWj0V1y90SiX4TUMgUEhiqxrtWQgsRQ6lBXByjD5ioVQjIvlouhrOk+HapymHmlNzD0Cf9nOltadyLxp2E9DjPyZlOGdFJ46H4ij/m5MWKE/vckbKLoXLz1TWnc66MpsUk+i8FqnXZlFq5p+KxdSPI38GmxU4OM2fjHsc+yd7fGLLTEMg1nOPMqXdzynCOcDbILh6dm8ni5ELO0VcEAXHdL+vAWeFqWeib8LcUe7NJwrozPpHuQlTkNwasRIPFOp87Ifcw9kmd8xubnUEQfdOQqceZa7bulOEcJFDoqsWxRL1A9o0kZifqAM64FqFRywBG2DoPGx8DOgKwprD+L4WOCVLp1rjUF2t95ozc49gnwd3Zt5Ww6dT7sOAyc5PP0dXM5eEcBrnZYyrnQrWbAp0D9a6b4EMwoTVbS5XMXOhcD5TYOiLjXAVIqYMj+YVBNoun5MDOd3Qd+n81jYubq8bfr+xJFY7zMzVdlzy2uBZi813y7GSNsb5WCeG+hxKtRQzRmVY9NeLWJYsLVZTfTKPSuFMV1mcXr7UAz8/UhrFPzsrEN5io6Yrqr1bzFthmcva9cP757vL2YnPbrhdsaa9W3BpFfVUStWqSDSCeng1Wg9mLhxuE1iNkNMUGW2xj8jn3FtvsPWWP43q+fP6b1ebu+lrEKrL6vPrURI/Eq6v1bcvr9U+iJhatlZ82wx7UVbq5EbnX89X/FN19TldfnlrKv375m93dITPzo2MTIY8j34+hXyXqpPNeJlKOtffeCqK1XkgRFEdB/Iup5uq5xdIyu16tZEukC2CJMNBiUcO+qMvlXV5dXF3cXqRLEfmnVHVf7+b2ovwkwz9f/cun9mX1MYll0+O3q9v16Az6EoF6kHx/cZ2ubjdbWnhsfNM+p4urXT3YMHP+eZYe4Jv1gK00F1FfWUzRx87OS6paHccCDYW4Ws4s9MUK8HUx/siJnWQzQeemsiuL9YD7eviYbmq7Ekz5JLABvLq9SV82o5TL+upWJDignVmJLETQI3aN/3SKfdL+/vOWDh7aXt6uRaCf2s3f7EWdmcgDbpYe8Jv1ULyxlHwKgUVGKQmrcDmQT14CTcQmSJShNVdRmEW2kQ2kKlqJklhUxLBYD/QMesqn9fpSZH552YZdZmRXJV0J2Fyt/nF9VdOXM8X4q5Vf5fUf2wBC8bHFv9y1jTTZUsFw5eJKXOY2Xe6KP85L46KfJfxvx/2IKTOWzIJCQtDVsEXk6L2YuyRxooKejKRvAvYQYxcaTwySsFubRCfQFgvfTuK+0LQt3Af7BPx2GfADzpsCBzSvBP3R1CyI0hMUfcvRMEu6LHYvITexx9glNBSJuCx2n5rxbNCZKmy6BAI2y02dZ0A/Hob+cBD6wX0V9r+f+5r1++PvWZ8S/31EF0niLLM33aZgTEo0vCdSMGMtTpDHeqFBJQDqm2c9tRprsp2c57xYGe7r8J8PwD/xV8I/zFwWAjCvBP+9SzLJulPWBt99d82aXMjp1k09O8BGyTQSUNKNHsF4SUr09UsJGMVlcafFavCT8G/5MPxLJH7Ef/5K/J9HRiO/Ev6T6UghxhS7Da0L4MdqkZzwfH3NL3l2JojN99a8UlKUixItEJDQSaa3WPphEv/9Du8HesJ/vxD/nZuZWbtXwv/kxGh7YkmrDEqEbaGInXe2XiyfyAnpdIUNFYAg2ZYJQjyTII+xklkLN1os7TgD/wkP4r8/iP/8dfA/7yCC9/RK0E/N+uBrKNlZb4ZJycoNsg0sZl0IS2hVtxDmgJF8aTZ7RhPY2gZuOeUEcxT6X0L+cAD5LXwl8geYuYX7lXAf9cwS1PciSPg8JtNKyZmLYyoaibmA1bMhNA+rLEFAkjJKYNkUqNEsToMBJnGfX6D9/AT7gF+H+wAzZ/nAvhbz5wjGlpgQbdH3vXsI1ZkIScKv8FSCoAcbUMix9RxNTODJZNO9beRxcdwFnEZ+2kF+eEJ+Woj8Zibym9dCfgmXURgNRerRSRoQggQAybDEvnututuNrCvWekQhRD6yJT3KwOuLQjYkv1jaNAP5/WHkB3OY+n8l9vuZzN+/FvEXRk+C9QIyWYiOtx2KqEByr8LoAUBgKFhJg1Ff4gJwUEVpsTrJjSGUsnjiB+xR9PcvoL8/gP5IX4n+ft6qksdXQv9aQTIsy93l1quXXBcZIEUQWgTdOZOrnq7SC4oqOjfncwzQkyHw0Csv1gFPo79/gfXDE/y7r0R/pJkzETQX/VW2DcyHsvlZ2z3J4ky1cnazvrttNx9S0V+ebdZ3N6Xd//lwnTabtnn4tGlic3VzJgqSe/2pPf7jw/AgD633vn341aap9ASZP6jvb3569sVuLy9cfejtpn1Ut5Bn3/rnbg/Pvn/87fDEDw+++5vt7+7b/3glBoPxBw08cnnvGBh39vT5+TPCWbDmzJ5b8PJvcMGc+XOOFLf+oxdMpDN3Ti7IBxSSL9+cG+d3Lh0dR8CJcUQ0Z3TOYjbSpw7Kncfgt8fBw928OeNzgLGZjFqb7V45Pg47MQ6H9gzP3fDUQpKlf5AHNGdRnpbP0Vv5NxkilYAPYfvK0RtHmBIAowpA4vgggKBPZmBfEfKF3EwSLP0QpbE/Vx1vXzk+jDBlD87KMCQ5Vk0bEYY/t47HG6DcwBKbrf/oFRRuexbPg3u51bExgaGpMYme6VzgcrBR60Q0FvVuKnp7LrR7uCBKcyLAsH3h+I3DPGOgQfTWgjxZwB2jVP0PT0xi4WoZltUymO32leOjiBOjAJGgPL+3am3gjPqG1e/FDuWfxvhBV+xFVx625Q47rY6OAqbs0zsdhKHhzioYcXpRhYwIUR2A79UzXLG6s33bSrabHR/HJGBFJ+MgP4zDkxoD04BEQOoBOCiFCMSC4RxG23i8dPTeAkGznMPH0RDVRwlxuIEIWVTNYd85wAZtZiC83Oz4oKaQ0wumWUHBQSAsEB3EfYcPQv3kg3UwOKkZBII7F/bvvPMemTt7/Pj8vqJQtQU6DyP8EgV5MhcGVceocSQEr5qwqgkUy3E7l47dmun4rcXNzuA8iEOKL4gXi3Ogup8V75MRkQI2OhG8uKL47faVo7e1W7d9ZBHqc6Su74f7ofUkaAcDBKN3qDfBqJesEesFe66Ot93w6D3d4XuOpmZJBy9OocLFewBARf3xsQglyxateow77Y7eMR4XbhTpi0AZB6fl0WntHuiBOoEbDE2dDVlVDDutdpsdG5EY6HFLYxjcZriZwAiJ8QrO7wCdtpJALeHQqvYpCB4DnTvmF5sdG1KgF0zBSzAUfBtUgWbwIwxm7x44OIWoRV/j37lixffURHyk/d8c7OzoGMNLY/QKOdarLyKQIsIwXgoCP3LzEAaXAMVyq3F7q9mRG4J5yT+8VSJm3Oh3ovIoDMqNjuoG1KEw3BFY76ga3G549J7+pXsOuMpm6MpIpA2CxXaEGC/9Sr6rt5RLw0MOl57aHbslwAu3tGLmj+RTXU4oGusHdIIG8TziIGWvVE181+B+KH78zdHb4wu3D9KJu4cECepRBWuGW9qgrEeMB0fF+lHNxHvuuPWr/RHsbCuwZ48fDxIS8Jo0SPf67CR2FpQXDU8IUcHJ0NYV4GdyeGh1dBRhYhTiMTKKkfCh2JbcS7KE5zJXdwcXn9FWCupwcUhqXmh2bHxuanwe1SsoqGCI3QCRIQ5OKTRtoIkQthQUhnbaHIX6h5fbHRtVgCmpiSUI/gTVHZHeFB9SOc2+xKUibl+ybufS0Xu7qXsPyaYzg93YwWdESoOsWMVPdl99EsuMOrOzL7Y6OqI4NSJQafDAJgloQG2MI3uigd+NakEcHMf5sHPp2L2F6MySBvstafBz+x2fWRmPtJKhRUm2/c6Vo6OYsIdolWc4N2hBSYf2OACc2JzmO2G4os28EG3euXL0xjjx+E6oDwsYDqInbwZrC/uQhR41AxKwGaySabBKjvByu6OjmnJaYlY2PegZnRsS20EApOlVYLs/JTHwTMV+fLnZkRGBYNHEiAYTDaOJkhv4Xhzna+Q2YofODzxbJ28k7OCYaTxeOnZvmHCPaDTxdn6ck5AET0zAPLNQBQydhhlNyKtR8v0EysOVH4fZwQbw0izfwXm9fnGzuf3wOd1++tCubm/W118OfLU7QXabLi7ZPDbf/bjb9OPlOsunh6a7H3eb1osbEc2H9VX7cHkh/wcSdQ98t/OjB7m/fBL2zrSHGi+OWf2IuG5MMKx/mImZ7pB3O4w8pqZjCowwWCuOeBuf3OTlI6Rhf4DGDAOMZsy9R0h6nJaY7tDudqipvA7QDf0NeeIZsrknEjTdX3xBghSeJEj3X/onzvPyCc57Ehyd7qHbITyoBMf+zPQAfdjtz4f78dHWnzBqhN30+ILZ0zCaJ9WKAAduweOnYKcVEmhfIeNEShgtxg5xGsKo7njQAndmDLaTpjFLN2cCFwMnHbIMM8yYSso+2RXvdGXd0JXXrtzQlZidPq130135na7kQXhg6fdkffjD6iQ6Ps9T3THsdqfr3PpLhqc/DCPdtnFydDv5/5DWxGEmgbb/RBvH7IUmu4v7DzvEhjikZoRDtsQeBmiBJ+704ukie6MD64dEKJjxYd1Tr2B5urv90Q0ZhK7PD/MloE/JYZjKUeCb6s7vajZ69yTB+z/CoId5DEYz2VvYHZzSMzNO+2o/oxn7wRB1lnOquwD7zzrYiR45prnvwO9JZ/A1+YaD3e3sf93mcTrB7UYQ8iMij8g0zjjGwz6x05vf6U3UOiZBPPbqRq41JgJs3WR3Ybc7vKeTZux15K6BB0LPzkx158xudzrzLb+M9xyYFI9oDB5C9/xkd3a3Ozfwe4x+zAKH2X3SyUT5Y2eMbu9hiczIP80QqI1/kh0GF6e683uq8OPspS666B/2I0oNowsAU90F3FPFaBMeR1Ix/EGllCpJpMnu3G53cN8Pj8yPRwY84snWEtiL3fk9K+axt/BEfBCsGZk0jKyxCycUSvrzRfvlgT1ubu/ql7N9Dnm/RHx9c/E53Xz5sLkrpW024oWfry/bbasf0tXml3bzeGFcrD67X+Y/W99cfNTZlg/Ckq/vbs+ky3bd5P+ubj/c3OWbiyKj2Mjoz1L9j7t6UYbH+7C5Tbd3m7Obu8v2IaeN3mbr8o9Xv5UUfuaU86GpT4lw5+bsx3f/psf46MlGSP/+4zv54l/0nOz15Z3eRY8Wvl2v7jbjG3z3T7R6+MlwwDMNK/pydevNwB9//AvUN3WRhsN2Vv9t5d2PP/7l+epHGfj2238/6IV3MpqtxyyfWvlJ/n66+5yuHhQknHj9+bNKrcoYf7v9EmLTPR2XMoSb1lbX7ebzxa1uVngY7O4LkDe6h0R+ca+j4RGk1834FmRaPSh15x3FlRLx8x/fTQh9d4LvoNAtjkIf3s/Uo7PcbKE//GQ4udwdFvr49qTV/3MidIsvCF0u/JcSOpwtiP8HsruRaN7buxmPuoAF9n7/k0H0L9m70ZPExtMoROpnq18+XZRPq/aHu3QpyvPnz7UA/tu0gOb//Z//C/QrMDpuPYXtUS/DxsGLLYWMitAH+LT3Au4Pw+CWi/lQ7jqsgxyR8/gK8VNqsC1DeYQ9MYoFi4jEglV0P/6oW4zqn7cb/Kd0bL5efhdXGz3ge+h3Jc8r4K2jeBresA1wGLO+OX0rdvyHO3nquhpmz1W27UZbrcYAIF/rHk3tIm/E9P56Vde6B3M1CE1ffH4MGasxFgxy11h0jybfXnqSTLK9Gdbz4FJMBN2CJa0SACmaVJMDExOGYuSjHjRhyVdb9HyhrqdDv7hr7OCuj71N2/+6vvlJn+zePQQCtmQ0xrhVX9/IZZFRua8coEZ2uEwD3lc7s3ulEsnsviL4ILydoznN/vuB961OXKHBRu4We0xCoShnDqk5KiS2JKkRsyefa6IeTag+e66maGk/D3pEK9q6UN7wfeS9VbYB3FgENPD+aYmw92LOIakPrablfoKKDZXBhVADGQiWquPGwXk9GhGTK6HIn0SYqsup6AkNnZzNIZueq6jryO7Ig3LHU8n9xTIO4WClHbB7JUrYTuqA7bQCTlC/AbAxFPlfcM2KIYtNZ8ETcl2ofOk1dpON9Sanlm1xbETuxifdvyMZDfZlCqBTKeCxqIOTQJK/PBRUnKjqIBnntO0HMy34U1Su9DFwF5nr0WEpkM1WMJ8p62Fi7AtSECbaI1UjTmK889KCi4/oJXu0dpng7XdFeEnJBvF7d370FfxD8n7++v33QXhvbWzGNPIEYtypFdJVIu9cApW4Kd0QiRqyqc1Ieo2uGz2OE0squcZl8ubvjvB4XzzN71fmAUSeIXXE10F4X1qApFGToLmWmVzkkgTBMQdncmyoRbwkYKHAf+hi+pl67UJ3Yq5H3gQ/vJ/5uyO8IM0BhEdr9qEGZkANvArGs7VUBNcNBaw2Y0idJAlwFplM7tzlYmx6Bj2yJZu84+RRTzfsPR579fWgCvypMZ5lYIrx95RmAuMR47T1Pztt6DthfBLOIgTdm4jJUgtcCSrVXmqjwl7wxQSqtjXIgWIsenqgIBFj9sLx0zLBh++L8WEsnOT3KruimzZ0afMqGC8wbYoE8EJ6zHa1QmnYYrAto1AciYKJOFDqrF/HnATefSk+E3CTKJuXyTt+f4xHHln8vsyFys7AeG31KhivJ9p3Y2rUGq9REvQIIXQtAs0OberNN5Na51D1lfBKWYJ+z7HbbqJkUcvsHMz3x3g+jPG4j/F+Bsb718F4J3FTaKLETl+g1KDL29FosWKJua0XTBAle6qlUkm+RN/0DcAUvNGymcvoDcCpMV4XpQaMD7N4vJ/mldLmVTA+QM+MNTjIjcCUoBVbMHMRkqOlSZ0N6g4+22ShBM7YesstYW3osZtlgsfvi/HOjnizN1ODjNNow/gqGE/CSWyuNUjY5ABaK8W0ZpxzOQfwho3kTJ67j1biKGn1OwkDIn6LUHOvy+RN3x/jYaQ1LuzP1HieYeXa6lUwvrfWYqm1OzH2bKtpPaEeZkDehJ4E1CkBdYGbYgVorA+2ukwlph6MBNtlcrffHePtCxj/LNTaGYH2deZqCvkmduxrtiZbJ3FWyE2oWYy9h95rJidcx1mItXM2ktFKPhXY56Zvfrdlk2XAJ8b4QH4oksyzCnDCDFoJc2jlCSA+UzYcjMq+5WjFzmtvvULSaj6UvRWMoZhs9oDkskVXo3clEkT0CXCZ3N33hfjoD08b+BkQ718H4iuiL+hMKiULZYwFfNXjtz1iSU65ZoYObLJB70BrznILQjuF+UfR1UIu419hqmYsosz7cwSWzIwJSW31KhDfEMmGKDwRYyndZSMixWJCi62Is4EADghrMd2nWFggJkAMiYMztkFdSOPDd4d45w5Oxsd9NeA0xAO+DsQ7azOxj2k4Gc57pTLJdGE4LUuQhYJCYVKvpUgwtrb7VnR1iqlVFtRvy1QQTw3xGKNCvKVZEB9pWu6RFkP8SweBs0QLrfRnjxV5MMKxxdBj7KqHbLvIO4tgA2t5Pi9xlZvpPaQsYJRdFwqUMGRhoiRkFKex/isqKz8NfLLCQ7Rv7CBw1hIPbLTGg52emBt2sh7W9+LiDiSWQxKwCI6x2uxyRMfWN+ygHtdBUglTUSK8h861B2zFSSJRxRuN92QRJQFhbgKFaZG659ZweRz45EH/Jvg3pW7SGoyku5doBn0zL86NLK7qoGajxdjpWNEe9h3Z6RyhrQi5S1xuxbTknA9ZN2hyyUkPoE3Jtxpj9OipS/iVRFM0bxYpe2bxlqeBTxdedG+rfIvVQh5WKy+SnzHl7tyksmdWdFAspHC2cseKeGAVapcaZyc0MumyX7RFtKv5U43QBGqog+g/xwzi9Y3IJScgTrX6HHmRsmfWbHka+HRFLnxjOK71FikMBR1m0KUXJ4MWV2ZSNLRiOvZoEWWgFEq2UUs6FF21csKQOHXoXaK0OHvTNV+tt2aFumbTq2Pvq0loMMAyz7bzYXwc+GQFZXBvrRqXbsyzQ51FO52h6PhPFbUti3uI8fCxEovcUuw9pSK5iuiRdVmBwHMGx6UaraRMLoDQsliI9YRfsYZikbrQ6pRgkbpnlmp5GvgkkAf7toqpynD+SsauRG3GqlKwfKqo7WjEQ3uMomWLknZK9mkjWnSdA0eJ3eLYWGIqNWWyJiTdJYnQKfSOJtiSKwmly9YvUvbMcslPA59UNr2x2opuoGhDNaYZC7fEdLKorWXqWKntsSlOQXFJdrvWTtWiJSEkjk2z28LFd9uMbYSic2Hr0etJFlq6JMknrZ1eyyJl+/np1zjwyagN4Q1mXzRUSabp+T0Z/qmitlZnVDQ8ysdNz8XGABK0u6bdtXOUDFvSad0C1Hrzwst10rVKaLfOcA/ZZzLA8pdaXKTsmbWRnwY+GbW9f1tl1v7CKkdzqFF7BiPX8Z8s19YZCq0tjEfTL33fTRh3MYiixpxDtJJbpaRv2UbJvkr1TUSSE8boeul6WnnTk4Eb9U6L1D2zKPLTwCd924e35dykVfV4KIs8w7X1OKQT5tqD3I4xchMy5dByZYnJ4qwu+VwGYG9VMi7xZIjWSdodvQOTnLD2rPUxoHGrhtwSZc8th/w08EllWxffXq4tGn8vg5+xYuDiqaK2FSzUQmX2WK7NJrZWewm+iYt7ya5Tbzn2KriO7CTjDmBT1vd+XYk9tcDNcc3VoY8OF6VfcyshPw18Utn01nBcqbgdcm07I9cme7JcW3MY6Y2PzZCTzV3IOLNLCbMEYqq6Lh26C1Fy7lItJpdDomYimaZHvPUubK77lE30i6L23BLITwOfUjYH4exvK9fmYU5FozZPz5Dr+E8VtVnx0E9E7e6zlrgttTbJwqxhCq6a0vSUcu5Okm/5IpeIJaBkWwmpBGjeaVmjHPKiGfK5tY+fBj49aUpvK9fmAcj9vKitoz9Zri096TtFFI5t5LQQY/bG614qI04L2JsJppXYnO5dVpxvNpERguadmAZ0x61HYs7C5RYpe+Y82tPAJxm59W9r1tTpxApbnSEP03zcejxV1Kb7+Ymja1+Yo2CzzdwYLeeAzecufNtSZcm1qEPOyTh92xSxW2letDSEkLiWKvtFEytzCx4/DXx67Qvf4JzpOEM+a/ELTxm1BzQ8OmUafRO6lVo31qoSWTLpQEK/5DMAlxypu+hLroG9o+LJkSH2elbR01TJPGW7+VF7HPikZ8c3N0OuUZv8ELWnJ011/CebIY/3c4907K0r0ndlW0CJx1E8umVCYyS5ttFQ6U1fh+iYSxY/dy5U6mitRGwxkqAcfpG6/YL0axj4pG9Hw28w/dKjUOycPSvmZFFb59rJCbk96ttUXNaynpSKTUDospOsC6hELWAIoRsXraTdtUKgKEl2tg2SlhkTGm+WMfIwfzlkHPj0Uie/weUQchK13YxX5fWMo1Pl2roPQAKgOxa1i+79BUmvumg9BuHZQK0CpNJCMEUUC2hsAJeiroDb5nI1LuVcQrXkwyJlz5xFexr49MTKG8u+rFJxomFdG+YcjLA0an9NwbpsUolCvFPSal3ZOi0LTsEolgN5fa1KwLonge4kCB+pWo4SwbWEtYPp5c3tI7eeFWy8XH+80Dpnm9Wnu7zayLPf3p/woxMloqjSLrVoXVzd3txp1braLkXkNyvwD1cHHT0UlXr6VnS80gNVVumXnXpSh4qp2Rnrzta+FFRPWdDO+6JvsVGxbCs1g9yopQABe8xVJzVMEursDGqJGRxebs5expaNHiFeF2ljv3SjyEqLpl1trtc3tyLp6/WwO1MEqcIE5NVGXWeoVydK0R2cekyQnnx0X8VuqOH1UNkOSVxyOFxIfO5GWMDFUP/6pokES/vcdiutaceXrd/ulblDmM5ctdGkbr69wJ1EOg+JsMUYtHi4M8Z58F1SVUDrWh6KJ+veI3DaUFebGGNnMeVemBfp5llZx19EQJ/WKuuhrLVYetFyXZuzB9FuVmhXH2/Wd9djSc3w2EDlv/l0cT0Uuhu/PH/ayPx4LpGIYG/6AKffCZI2k6L/9rp2pZdgkoSf2lOyuaYaU5NkUWJRaeib6KD3UKik0iUARU0z0/CSkM+lz3i3eees/33Rfxap330e5U5xNYyq1dXHu4uqkhSgul3f3dyXGlQNId5f/Gs9deqq/XL/cTDyewTbKbp5f/lgoUeO01rQRpNq+PZYUTwmLfJLnKw8rCTovYLWeQRh8cEPr5tXIaTCBnIhi204RKdV0wugRJVFarALYsVQ1PQ+VqB7FivghLGCZhzrQuFVYkXNIbmGiJJJ2Va5R5O8qdFJ0m3ZVDClNshQc7ZosfqK1ku8b+zSUqfghbHCH4kV9nmsoFPECsLphWht9Aqxgn12ujnQoa5OtGAkURJmRcZ67LmVlEEoVyMjOCZhWzKpLuEjGky2mznTXTuHwE7ECo4HYoXZjhX+m2OFnfGWtPWvESsIq2BNkQzUlcitFdejBGCKwXrWF4rIcRL2ZFv0VFtz+l1iQ9n3nmecdbRzYO7RWEHPYwXuxwphTqeLFUR2hge8uIZ3ylhhgxFugFGLMHdwIQbJHqxNriP7ppULXTUomaItegoSJBOInehKiKyvARepISyIFYzH8opwwlCBM45swOBfIVTou1tZT5zKEh04S7rHvQkg2W5jCcwSvq31GVmidhYPAXB6Hk+JRDX0GBYpIy4LFd68HCr8d4oUTDMoFb1CnPBCXMH2ILFb6JT30Vo9A6yyzpNXNF68wyP1Fk3UvWl6jAmLTrRwgMO8KPsGMxEniJ7HCQGsrTgRvzlOeJhenPIvbg08aU5hJCBgZpRETd+siz1BRm/l29YYvG8kKQWAMdmxJn0kMRyG1Ug2LdIi0cOxOMHmeZwIz+KEO2GcEII47QD2xb3Xp4wTtfXsKqHRnVoSmBtlzs6hROxaiJtkElwzRgEuE3SlnwIGcLGx7cmmskgNuCBO+O2cAp7lFKcMFDxjvywTvkKgCOyTUFZJG1jPjNFq8thtabFzyMmFhl1ScMkCDceSrMRqlObkYqwsrrQoUOxvwJgKFHwsp4gH5p/sKeafIsyYf4rwCrEiRFEJlxiEv4YAxYLxlKhoPTQo3UtebnuEhJKtddGaRWG6YJ1kG6Y0XsSowE7ECnNg/ol4K1Z8e6igGGesdcZXCBVZq4o4F6vYfQLwKQtL6sYLw2Xg0CU6hEQkrYr3wmO1OBIaax1pxkfLMIqPhQr7fPoJn00/CQqdLlQIS5zhAPZVpp9yTqUHSK4mI6E45KB1YIXPesEf0jOwJG4LkzURavEhtIjoe5HkAnX+fJEa3JKlCnc0VJx0qeJEefa3h4pEWnNJEmffVepEXLGxEFqgzsV5tDGjhHHhsPq2qxJcnwJWnTU01i+aftpf9Z8KFYaOhIoDSQWGU0w/zVjB1UavMf1UobvgJEREm4Oe/ya5XrNRonsXzTTjJX604FIEq+eCZksCaTmIp7TuZyzSb+smTIQKjxNpBZpvTyvCdKzQIlzfP1ZIMM69BhbOH8QZnJVULqLQ2gJ6wE3ChuIGTBK7Y7EcooGcqo/V9QZzjujbFn08FivwQFoBz5YqzCnTihkTHtpoTqxQEcsNL3rb6M/+/G6sqDSI/52u0rebn4cFhYeq1PrvsWrO+K+P7aqpAb3Tnraqk3wYVs8VJsPehc2nhOxGlBWnEBfyHpzXLacGU60tMXOHblzvOo/ltJJAK+JZRFGXYZMSrhzH3LCsa/ugVUQu7o+0Q+QUYzOS5bTeiSEY24KH6jkAMFff0dedR9Lnznf14VggHXBt2vGNdqkYNGDFgwS3niCRvuumJ+aJJTpfQ8+9xTLs0KEUomsts769LP8NRRh+rcA5Ry4SUTEOlOWz3Ojy3gjT5/Sn9dWvf/O79d/97h/fw7n/708ttGaK7pkYkCeE4qtyUsnUmtHXMqKG6yyUKOSKrluX5IEHw7m4umr1Q259fdM+XA1lV/TcH9WxxNA22FYbRagY/OFqPRqMloX5LZhf/xZ0D8j15VoUuL75Iub7Md0O+PJ5LX+Tmv7P+NerYa+HHjYkfd2btpZYEfuvWrqnX9yuLiQgCCDdCrCs/km84FbsaPVgpJtfS6j5eB8e7q4u/nA3HLmlPrr6h78T13lo+L5ebP5jfaGhvNysNxu5+2rwBmmz+aJ7TrSAjfb9NIqkRWUeq3utcrsqnz6nm5/O98x4ri3c+4A2L3ej6T0WN3q3+6P7UkeDJtMfPzz2Hp12NNa/3b3vsMx48LZDGTT96p/bdbq4+X1ZX7f//f5/jYmtXvtw++V60J4q+lEYHx6wrqw/SSgfoEI8/tFJ77cyyjcPMhaDHEHg4HlwZy+cV3b2wiFaZy8c8nR2eNPk2eHtdWeHN2KdHd6yc3aYkJ0d5gJnh+PU2WHcVLATYnRx9VxeW1D3TGYvXIMj1/DINdq/tifDw5fg5Uv48qVn99qT6eFL8PIlfPnSKF8JiQ9w94hPAjXv0aB7D0ayofc/i1P85/8Hlos2kw==')))
OUT=Path('/content/RSX_V2_13_CPU_REPORT')
OUT.mkdir(parents=True,exist_ok=True)
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
manifest=DATA['manifest'];cal=DATA['calibration'];test=DATA['heldout']
assert digest(cal)==manifest['calibration_sha256']
assert digest(test)==manifest['heldout_sha256']
assert len(cal)==48 and len(test)==60
assert not ({p['problem_id'] for p in cal} & {p['problem_id'] for p in test})
assert set(p['template'] for p in cal).isdisjoint({p['template'] for p in test})
assert len({p['prompt'] for p in cal+test})==len(cal+test)
# Mathematical oracle checks: all are generated with symbolic proof; verify three-number Countdown safe arithmetic.
import ast,math
for p in cal+test:
    if p['kind']=='countdown':
        expr=ast.parse(p['proof'],mode='eval')
        literals=[n.value for n in ast.walk(expr) if isinstance(n,ast.Constant)]
        assert sorted(literals)==sorted(p['numbers'])
        assert eval(p['proof'],{'__builtins__':{}})==p['target']
    else:
        assert isinstance(p['target'],int)
review=pd.read_csv(io.StringIO(DATA['four_review_csv']))
e10=pd.read_csv(io.StringIO(DATA['e10_csv']))
e11=pd.read_csv(io.StringIO(DATA['e11_csv']))
assert len(review)==4 and len(e10)==36 and len(e11)==30
assert e10['problem_id'].is_unique and e11['problem_id'].is_unique
review['independent_human_verdict']='UNREVIEWED'
review['reviewer_notes']=''
review.to_csv(OUT/'FOUR_CASES_HUMAN_REVIEW_REQUIRED.csv',index=False)
for name,data in [('V2_FROZEN_DATA_MANIFEST.json',manifest),('V2_CALIBRATION_CASES.json',cal),('V2_HELDOUT_CASES.json',test)]:
    (OUT/name).write_text(json.dumps(data,indent=2))
checks={'historical_E10':len(e10),'historical_E11':len(e11),'E12_scoring_disputes':len(review),
        'fresh_calibration':len(cal),'fresh_heldout':len(test),
        'calibration_sha256':manifest['calibration_sha256'],
        'heldout_sha256':manifest['heldout_sha256'],'manifest_sha256':digest(manifest),
        'audit_not_human_adjudication':True,'labels_touched_for_policy_tuning':False}
(OUT/'E13_PROTOCOL_AUDIT.json').write_text(json.dumps(checks,indent=2))
(OUT/'CALIBRATION_ONLY_STOP_RULE.md').write_text('# Precommitted stop rule\n\nThe v2 policy is trained only from its new calibration split. The E10 test outcomes do not enter training. The calibration router must have positive out-of-fold gain versus preserving, no worse out-of-fold gain than always-finalize, at most two out-of-fold harmful flips, and at least one active routing bucket; otherwise stop without running prospective confirmation by default. All v2 held-out outcomes remain unavailable until the policy is locked. This is a synthetic benchmark, not an external standardized benchmark.\n')
print('E13 AUDIT COMPLETE',json.dumps(checks,indent=2))
display(review[['problem_id','action','adjudication_status','independent_human_verdict']])
archive=Path('/content/RSX_V2_13_CPU_Artifacts.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for p in OUT.iterdir():
  if p.is_file():z.write(p,p.name)
print('Artifacts:',archive)
try:
 from google.colab import files
 files.download(str(archive))
except Exception:pass
